# Chaîne des Puys: olivine exposure series

Updated for the refactored `primus_paleo` (November 2026). What changed with respect to the previous version:

* **Radiogenic background**: one `integrate_radiogenic` per sample instead of `integrate_fission_spectrum` +
  `integrate_background_neutron_spectrum`.
* **Track groups**: the integrals run with `nucleus='groups'`; the old total is the `'total_no_light_ions'` entry, and
  the heavy / medium / light split is there for `plot_track_groups`.
* **Zenith distribution**: cosmic rays reach the sample with the MCEq zenith distribution of each species (bug fix:
  before, all directions had equal weight in cos(zenith)). At 1 m.w.e. this lowers the cosmic-ray tracks compared
  with the old code, by roughly a factor 2.
* **Observable**: besides tracks per mg, the expected etched pit counts on the analysed area
  (`slice_spectrum_etch`), whose Poisson errors are the ones that matter for telling the scenarios apart.
* The Laschamp excursion needs the baseline and the events as specs (dicts), as here: their templates are rebuilt at
  the excursion cutoff.

The first run builds the MCEq response matrix for 1 km altitude once; later runs read it.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import mark_inset, zoomed_inset_axes

import primus_paleo as pp
from primus_paleo import FluxHistory, Paleodetector, plot_track_groups, slice_spectrum_etch

SUM = pp.TOTAL_NO_LIGHT_IONS   # the total used before

In [ ]:
mineral_name = "Olivine"

sample_area_cm2 = 2.5 * 2.5
magnification = 10

overburden_history = {
    "initial_depth": 1.
}

h_obs_km = 1.
laschamp = {"start_time_kyr": -42., "end_time_kyr": -40., "geomagnetic_cutoff": 0.}
sn250 = {"start_time_kyr": -75., "template": {"kind": "SN", "distance_pc": 250.}}

flux_histories = {
    'base': FluxHistory(h_obs_km=h_obs_km),
    'base + Laschamp': FluxHistory(h_obs_km=h_obs_km, excursions=[laschamp]),
    'SN 250 pc': FluxHistory(h_obs_km=h_obs_km, events=[sn250]),
    'SN 250 pc + Laschamp': FluxHistory(h_obs_km=h_obs_km, events=[sn250], excursions=[laschamp]),
}

In [ ]:
exposure_times = np.array([41.4, 30., 13.1, 10.96, 9.5, 8.64, 7.65])
exposure_times_err = np.array([1., 2., 0.7, 0.15, 0.5, 0.06, 0.12])
names = np.array(["Laschamp", "Lemptegy", "Come", "Dome", "Pariou", "La Vache", "Montcineyre"], dtype=str)

## Integrals

One detector per lava flow. The Laschamp excursion (42-40 ka) only matters for the oldest flow, the only one exposed
then.

In [ ]:
samples = {}
for name, age in zip(names, exposure_times):
    det = Paleodetector(mineral_name, total_age_kyr=age, flux_history=flux_histories['base'],
                        overburden_history=overburden_history)
    det.verbose = 0
    histories = ['base', 'SN 250 pc'] + (['base + Laschamp', 'SN 250 pc + Laschamp'] if age > 40. else [])
    x_bins, radiogenic = det.integrate_radiogenic(nucleus='groups')
    cosmic = {h: det.integrate_all_particles(nucleus='groups', flux_history=flux_histories[h])[1] for h in histories}
    samples[name] = {'detector': det, 'age': age, 'radiogenic': radiogenic, 'cosmic': cosmic}

samples[names[0]]['detector'].describe_track_groups()

## Tracks per mg vs exposure time

In [ ]:
def tracks_per_mg(sample, history):
    """Total tracks per mg of one sample: cosmic rays with the given flux history plus the radiogenic background."""
    return np.sum(sample['cosmic'][history]['total'][SUM] + sample['radiogenic']['total'][SUM])

summed = {h: np.array([tracks_per_mg(samples[n], h) for n in names]) for h in ('base', 'SN 250 pc')}
summed_laschamp = {h: tracks_per_mg(samples[names[0]], h) for h in ('base + Laschamp', 'SN 250 pc + Laschamp')}

In [ ]:
def age_boxes(ax, values, color):
    """Age uncertainty of each flow (grey band) and +-10% / +-30% boxes around the predictions."""
    for age, err, v in zip(exposure_times, exposure_times_err, values):
        ax.axvspan(age - err, age + err, color='grey', alpha=0.15, linewidth=0)
        for half_width, alpha in ((0.1, 0.4), (0.3, 0.2)):
            ax.add_patch(patches.Rectangle((age - err, (1 - half_width) * v), 2 * err, 2 * half_width * v,
                                           linewidth=0.1, facecolor=color, edgecolor=color, alpha=alpha))

def draw_series(ax):
    for h, color in (('base', 'C0'), ('SN 250 pc', 'C1')):
        ax.errorbar(exposure_times, summed[h], yerr=np.sqrt(summed[h]), label=f"{h} scenario", linewidth=1.8,
                    color=color, marker='o', markersize=1., linestyle='--', capsize=0)
        age_boxes(ax, summed[h], color)

fig, ax = plt.subplots(figsize=[7, 5])
draw_series(ax)
for h, color in (('base + Laschamp', 'C0'), ('SN 250 pc + Laschamp', 'C1')):
    ax.errorbar(exposure_times[0], summed_laschamp[h], yerr=np.sqrt(summed_laschamp[h]), label=h, color=color,
                marker='d', markersize=3, linestyle='', capsize=0)

axins = zoomed_inset_axes(ax, zoom=2., loc='upper left', bbox_to_anchor=(0.15, 0.6, 0.8, 0.4), bbox_transform=ax.transAxes)
draw_series(axins)
young = exposure_times < 15
axins.set_xlim(7.3, 14.2)
axins.set_ylim(0.8 * summed['base'][young].min(), 1.2 * summed['SN 250 pc'][young].max())
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5", linestyle='--')

ax.set_xlabel(r"Exposure Time (kyr)")
ax.set_ylabel(r"Tracks (mg$^{-1}$)")
ax.legend(fontsize='small', loc='upper right')
plt.show()

## What an etched sample would show

Expected pits on the analysed area after HF etching, at `magnification`. The SN signal is the excess over the base
scenario. The expected pit count is linear in the track spectrum, so the excess spectrum is etched on its own: its
Monte Carlo error (~2% per call) is then a fraction of the excess, not of the total, which would swamp a 1% effect.
The significance uses the Poisson error of the base counts only (no systematics, no age error).

In [ ]:
def pits(sample, counts_mg):
    """Expected etched pits on the analysed area for a track spectrum of this sample [per mg]."""
    return np.sum(slice_spectrum_etch(counts_mg, sample_area_cm2=sample_area_cm2, x_bins=x_bins,
                                      sample_density_g_cm3=sample['detector'].config['density_g_cm3'],
                                      etch_type='HF', magnification=magnification))

def spectrum(sample, history):
    return sample['cosmic'][history]['total'][SUM]

n_base = np.array([pits(samples[n], spectrum(samples[n], 'base') + samples[n]['radiogenic']['total'][SUM])
                   for n in names])
n_excess = np.array([pits(samples[n], np.clip(spectrum(samples[n], 'SN 250 pc') - spectrum(samples[n], 'base'), 0, None))
                     for n in names])
n_sn = n_base + n_excess
significance = n_excess / np.sqrt(n_base)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=[7, 6], sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax1.errorbar(exposure_times, n_base, yerr=np.sqrt(n_base), xerr=exposure_times_err, fmt='o', color='C0', label='base')
ax1.errorbar(exposure_times, n_sn, yerr=np.sqrt(n_sn), xerr=exposure_times_err, fmt='s', color='C1', label='SN 250 pc')
for name, t, n in zip(names, exposure_times, n_base):
    ax1.annotate(name, (t, n), textcoords='offset points', xytext=(4, -10), fontsize='x-small')
ax1.set_ylabel(f'Expected pits on {sample_area_cm2:g} cm$^2$')
ax1.set_yscale('log')
ax1.legend(fontsize='small')
ax2.plot(exposure_times, significance, 'o-', color='C1')
ax2.axhline(3, color='grey', linestyle=':')
ax2.set_xlabel('Exposure time (kyr)')
ax2.set_ylabel(r'SN excess ($\sigma$)')
plt.show()

for name, nb_, ne, s in zip(names, n_base, n_excess, significance):
    print(f"{name:12s} base {nb_:10.0f}   SN 250 pc excess {ne:8.1f} pits = {s:4.2f} sigma")

## Composition of the Laschamp sample

Track groups for the four flux histories of the oldest flow: where in the spectrum the supernova and the excursion
add tracks.

In [ ]:
oldest = samples[names[0]]
curves = {h: oldest['cosmic'][h]['total'] for h in oldest['cosmic']}
curves['radiogenic'] = oldest['radiogenic']['total']
fig, axes = plot_track_groups(curves, x_bins, style='panels', logy=True, y_decades=6)
plt.show()

base = oldest['cosmic']['base']
by_species = {species: v[SUM] for species, v in base.items()}
fig, ax = plot_track_groups(by_species, x_bins, style='lines', keys=[k for k in by_species if k != 'total'],
                            total='total', logy=True, y_decades=6)
ax.set_title(f"{names[0]}: cosmic-ray tracks by species (base)")
plt.show()